EN ESTE NOTEBOOK:
- Creamos 3 datasets con las variables de entrada. 1. Variables de gestación, 2. Variables ..., 3. Unión de todas las variables anteriores.

*PARA CADA DATASET:*
- Imputamos con Miss Forest (sobre el conjunto de train)
- Aplicamos 6 algoritmos de clasificación (SVM, Random Forest, Logistic Regression, Tree Decision, KNN y XGBoost) con los hiperparámetros por defecto (class_weight siempre tiene que ser balanced) y vemos qué f1-score sale.
- Aplicamos GridShearch para buscar los hiperparámetros que optimizan el f1-score cada uno de los algortimos.

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('datos.csv')

In [25]:
df2 = pd.read_csv('datos_embarazo.csv')

# Creamos datasets

In [ ]:
# DATASET 1: Variables de la gestación (quitamos las tipo datetime)
variables_gestacion = [
    'id',
    'edad_materna_gest',
    'tas_1tri',
    'tad_1tri',
    'eg_eco_1tri',
    'uterinas_p95_1tri',
    'plgf_1tri',
    'eg_plgf_1tri',
    'unidades_plgf_1tri',
    'valor_plgf_1tri',
    'riesgo_pe_1tri',
    'tomo_durante_gest',
    'uterinas_p95_eco_2tri',
    'eg_eco_2tri',
    'deter_sflt1_plgf_gest',
    'eg_deter_sflt1_plgf',
    'valor_sflt1',
    'valor_plgf',
    'ratio_sflt1_plgf'
]

# Creamos el dataset asegurando que existan en el dataframe original
df_embarazo = df[[col for col in variables_gestacion if col in df.columns]].copy()

In [ ]:
# DATASET 2: edad actual, peso actual, IMC_actual, ant_medico_actual, tabaco, y las que creamos para dieta, ejercicio, estrés

In [ ]:
# DATASET 3: todo lo del dataset 2 + grasa visceral, ratio cintura/cadera, hemoglobina glicada, LDL, HDL, trigliceridos, creatininia, ratio albumina/creatinina, Troponina T, NT-proBNP, ácido úrico

In [ ]:
# DATASET 4: dataset1 + dataset3

# Creamos etiqueta (riesgo_cv_compuesto)
riesgo_cv_compuesto = 1 si ≥1 de: 
1. TA sistólica ('ta_sistolica) ≥140 o TA diastólica ('ta_diastolica') ≥90 
2. Masa VI indexada ('masa_vi_tdiast_indexada') >95 g/m² 
4. FEVI ('fevi_simpson_4c') < 50%
5. Cardiopatía estructural ('cardiopatia_estructural') presente

In [6]:
# Definimos la columna clave y las variables del Outcome 1
ID_COL = 'id'
variables_outcome = [
    'ta_sistolica', 
    'ta_diastolica', 
    'masa_vi_tdiast_indexada', 
    'fevi_simpson_4c', 
    'cardiopatia_estructural'
]

# Función auxiliar para evaluar las condiciones del outcome
def evaluar_riesgo(dataframe, incluir_strain=True):
    condicion = (
        (dataframe['ta_sistolica'] >= 140) | 
        (dataframe['ta_diastolica'] >= 90) | 
        (dataframe['masa_vi_tdiast_indexada'] > 95) | 
        (dataframe['fevi_simpson_4c'] < 50) | 
        (dataframe['cardiopatia_estructural'] == 1)
    )
    return condicion.astype(int)

# Función para imprimir la tabla de distribución
def mostrar_distribucion(dataframe, titulo):
    dist = dataframe['riesgo_cv_compuesto'].value_counts().reset_index()
    dist.columns = ['riesgo_cv_compuesto', 'Nº Pacientes']
    dist['%'] = (dist['Nº Pacientes'] / len(dataframe) * 100).round(2).astype(str) + '%'
    print(f"\n{titulo} (Total N = {len(dataframe)})")
    display(dist)

# CASO 3: Sin 'strain' y sin missings en el resto
# Filtramos para quedarnos con filas completas en las 5 variables restantes
df_caso3 = df.dropna(subset=variables_outcome).copy()
df_caso3['riesgo_cv_compuesto'] = evaluar_riesgo(df_caso3, incluir_strain=False)
mostrar_distribucion(df_caso3, "CASO 3: Sin 'strain_longitudinal_vi' y sin missings")


CASO 3: Sin 'strain_longitudinal_vi' y sin missings (Total N = 369)


,riesgo_cv_compuesto,Nº Pacientes,%
0,0,312,84.55%
1,1,57,15.45%


In [7]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score

# Importación de los 6 algoritmos de clasificación requeridos
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

In [8]:
# Sincronizamos df_embarazo para que contenga únicamente las mismas 369 pacientes del Caso 3
df_embarazo_filtrado = df_embarazo[df_embarazo['id'].isin(df_caso3['id'])].copy()

# Garantizamos que la variable objetivo 'y' esté perfectamente alineada por el ID de la paciente
y = df_caso3.set_index('id').loc[df_embarazo_filtrado['id']]['riesgo_cv_compuesto'].values

# Definimos la matriz de características X excluyendo el identificador 'id'
X_dataset1 = df_embarazo_filtrado.drop(columns=['id'], errors='ignore')
#X_dataset2 = 
# X_dataset3 = 

# Listas de intenciones originales
num_cols_deseadas = [
    'edad_materna_gest', 'tas_1tri', 'tad_1tri', 'eg_eco_1tri', 
    'eg_plgf_1tri', 'valor_plgf_1tri', 'eg_eco_2tri', 
    'eg_deter_sflt1_plgf', 'valor_sflt1', 'valor_plgf', 'ratio_sflt1_plgf'
]

cat_cols_ord_deseadas = [
    'uterinas_p95_1tri', 'riesgo_pe_1tri', 'uterinas_p95_eco_2tri', 'deter_sflt1_plgf_gest'
]

cat_cols_nom_deseadas = [
    'unidades_plgf_1tri', 'tomo_durante_gest'
]

# Filtramos cada lista quedándonos SÓLO con las columnas que de verdad existen en X_dataset1
num_cols = [col for col in num_cols_deseadas if col in X_dataset1.columns]
cat_cols_ord = [col for col in cat_cols_ord_deseadas if col in X_dataset1.columns]
cat_cols_nom = [col for col in cat_cols_nom_deseadas if col in X_dataset1.columns]
cat_cols = cat_cols_ord + cat_cols_nom

# Forzamos una codificación temporal LabelEncoding/Ordinal para que IterativeImputer pueda procesarlas
X_dataset1_encoded = X_dataset1.copy()
for col in cat_cols:
    X_dataset1_encoded[col] = X_dataset1_encoded[col].astype('category').cat.codes.replace(-1, np.nan)

In [9]:
# División de datos (Train / Test)
# Usamos stratify=y para asegurar que el 15.45% de casos positivos se mantenga equitativo en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X_dataset1_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# Separamos Train y Test en sus correspondientes bloques numéricos y categóricos
train_num = X_train[num_cols]
test_num = X_test[num_cols]
train_cat = X_train[cat_cols]
test_cat = X_test[cat_cols]

# Imputación con MissForest
Considerando división train/test

In [10]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import joblib
pd.set_option('display.max_columns', None)

In [11]:
# Funciones para la imputación
def _missForest_fit(data, num_estimators, m_depth, m_samples_leaf, max_iter,
                    min_value=None, max_value=None, random_state=1234):
    tree = ExtraTreesRegressor(n_estimators=num_estimators,
                               random_state=random_state,
                               max_depth=m_depth,
                               min_samples_leaf=m_samples_leaf)
    imputer = IterativeImputer(estimator=tree,
                               random_state=random_state,
                               max_iter=max_iter,
                               min_value=min_value,
                               max_value=max_value,
                               imputation_order='ascending',
                               initial_strategy='mean',
                               verbose=2)
    imputer.fit(data)
    return imputer

def _missForest_transform(imputer, data):
    return imputer.transform(data)


def imputacion_pipeline(train_num, test_num, train_cat, test_cat,
                        num_estimators, m_depth, m_samples_leaf,
                        max_iter, random_state=1234,
                        guardar_imputador=True):

    # Obtener min/max automáticamente por nombre de variable
    min_values_num = [min_values_dict[col] for col in train_num.columns]
    max_values_num = [max_values_dict[col] for col in train_num.columns]

    min_values_all = [min_values_dict[col] for col in list(train_num.columns) + list(train_cat.columns)]
    max_values_all = [max_values_dict[col] for col in list(train_num.columns) + list(train_cat.columns)]

    
    # Imputación de numéricas
    imputer_num = _missForest_fit(train_num, num_estimators, m_depth, m_samples_leaf,
                                  max_iter, min_value=min_values_num,
                                  max_value=max_values_num,
                                  random_state=random_state)
    train_num_imp = pd.DataFrame(_missForest_transform(imputer_num, train_num),
                                 columns=train_num.columns,
                                 index=train_num.index)
    test_num_imp = pd.DataFrame(_missForest_transform(imputer_num, test_num),
                                columns=test_num.columns,
                                index=test_num.index)

    # Unir con categóricas
    train_all = pd.concat([train_num_imp, train_cat], axis=1)
    test_all = pd.concat([test_num_imp, test_cat], axis=1)

    # Imputación conjunta
    imputer_all = _missForest_fit(train_all, num_estimators, m_depth, m_samples_leaf,
                                  max_iter, min_value=min_values_all,
                                  max_value=max_values_all,
                                  random_state=random_state)
    train_all_imp = _missForest_transform(imputer_all, train_all)
    test_all_imp = _missForest_transform(imputer_all, test_all)

    # Control de variables categóricas
    for i in range(train_num.shape[1], train_all.shape[1]):
        train_all_imp[:, i] = np.clip(train_all_imp[:, i], min_values_all[i], max_values_all[i])
        test_all_imp[:, i] = np.clip(test_all_imp[:, i], min_values_all[i], max_values_all[i])

    train_final = pd.DataFrame(train_all_imp, columns=train_all.columns, index=train_all.index)
    test_final = pd.DataFrame(test_all_imp, columns=test_all.columns, index=test_all.index)

    for col in train_cat.columns:
        train_final[col] = train_final[col].round().astype(int)
        test_final[col] = test_final[col].round().astype(int)

    if guardar_imputador:
        joblib.dump(imputer_all, 'imputador_missforest.pkl')
        print("Imputador guardado como 'imputador_missforest.pkl'")

    return train_final, test_final

In [12]:
min_values_dict = X_train.min().to_dict()
max_values_dict = X_train.max().to_dict()

In [13]:
max_values_dict

{'edad_materna_gest': 46.17,
 'tas_1tri': 165.0,
 'tad_1tri': 107.0,
 'eg_eco_1tri': 96.9,
 'uterinas_p95_1tri': 1.0,
 'plgf_1tri': 1.0,
 'riesgo_pe_1tri': 1.0,
 'tomo_durante_gest': 4.0,
 'uterinas_p95_eco_2tri': 1.0,
 'deter_sflt1_plgf_gest': 1.0}

In [14]:
# Llamar a la función para imputar
train_final, test_final = imputacion_pipeline(
    train_num=train_num,
    test_num=test_num,
    train_cat=train_cat,
    test_cat=test_cat,
    num_estimators=50, # Número de árboles en el modelo ExtraTrees
    m_depth=4, # Profundidad máxima de los árboles
    m_samples_leaf=3, # Mínimo de muestras en cada hoja
    max_iter=100,
    random_state=1234,
    guardar_imputador=True
)

[IterativeImputer] Completing matrix with shape (295, 4)
[IterativeImputer] Ending imputation round 1/100, elapsed time 0.06
[IterativeImputer] Change: 2.1518012944907383, scaled tolerance: 0.165 
[IterativeImputer] Ending imputation round 2/100, elapsed time 0.11
[IterativeImputer] Change: 1.3809722838248764, scaled tolerance: 0.165 
[IterativeImputer] Ending imputation round 3/100, elapsed time 0.17
[IterativeImputer] Change: 0.6785380966859123, scaled tolerance: 0.165 
[IterativeImputer] Ending imputation round 4/100, elapsed time 0.21
[IterativeImputer] Change: 0.6822053755916073, scaled tolerance: 0.165 
[IterativeImputer] Ending imputation round 5/100, elapsed time 0.26
[IterativeImputer] Change: 0.42265765395694643, scaled tolerance: 0.165 
[IterativeImputer] Ending imputation round 6/100, elapsed time 0.31
[IterativeImputer] Change: 0.09816099773242115, scaled tolerance: 0.165 
[IterativeImputer] Early stopping criterion reached.
[IterativeImputer] Completing matrix with shape 

### Análisis de sensibilidad pre y post imputación

In [16]:
# Reconstruimos el dataset completo ANTES de imputar (con NaNs)
df_antes_imputar = pd.concat([X_train, X_test], axis=0)

# Reconstruimos el dataset completo DESPUÉS de imputar (sin NaNs)
df_despues_imputar = pd.concat([train_final, test_final], axis=0)

# Calculamos las estadísticas descriptivas para las variables numéricas
# Usamos solo num_cols porque las categóricas se evalúan mejor con frecuencias
stats_antes = df_antes_imputar[num_cols].describe().T[['mean', 'std', '50%']]
stats_despues = df_despues_imputar[num_cols].describe().T[['mean', 'std', '50%']]

# Renombramos las columnas para poder unirlas de forma clara
stats_antes.columns = ['Media (Antes)', 'Desv.Est (Antes)', 'Mediana (Antes)']
stats_despues.columns = ['Media (Después)', 'Desv.Est (Después)', 'Mediana (Después)']

# Combinamos ambas tablas por el nombre de la variable
tabla_sensibilidad = pd.concat([stats_antes, stats_despues], axis=1)

# Calculamos la diferencia absoluta entre las medias para medir el impacto
tabla_sensibilidad['Dif. Absoluta Medias'] = (
    tabla_sensibilidad['Media (Antes)'] - tabla_sensibilidad['Media (Después)']
).abs().round(4)

# Ordenamos las columnas para facilitar la lectura visual uno a uno
columnas_ordenadas = [
    'Media (Antes)', 'Media (Después)', 'Dif. Absoluta Medias',
    'Desv.Est (Antes)', 'Desv.Est (Después)',
    'Mediana (Antes)', 'Mediana (Después)'
]
tabla_sensibilidad = tabla_sensibilidad[columnas_ordenadas].round(2)

print("="*90)
print("ANÁLISIS DE SENSIBILIDAD: VARIABLES NUMÉRICAS")
print("="*90)
display(tabla_sensibilidad)


# CONTROL DE CALIDAD PARA VARIABLES CATEGÓRICAS 
print("\n" + "="*90)
print("ANÁLISIS DE SENSIBILIDAD: VARIABLES CATEGÓRICAS")
print("="*90)

for col in cat_cols:
    # Calculamos las frecuencias relativas originales
    prop_antes = df_antes_imputar[col].value_counts(normalize=True) * 100
    prop_despues = df_despues_imputar[col].value_counts(normalize=True) * 100
    
    # Creamos un índice único combinando todas las categorías que existen en ambos mundos
    todas_las_categorias = sorted(list(set(prop_antes.index) | set(prop_despues.index)))
    
    # Forzamos a ambas series a tener exactamente las mismas categorías (rellenando con 0 si faltan)
    prop_antes = prop_antes.reindex(todas_las_categorias, fill_value=0.0)
    prop_despues = prop_despues.reindex(todas_las_categorias, fill_value=0.0)
    
    # Ahora que miden exactamente lo mismo, montamos el DataFrame de forma segura
    df_cat_comp = pd.DataFrame({
        'Categoría': todas_las_categorias,
        '% Antes': prop_antes.values,
        '% Después': prop_despues.values
    }).round(2)
    
    # Calculamos la diferencia absoluta en puntos porcentuales
    df_cat_comp['Dif %'] = (df_cat_comp['% Antes'] - df_cat_comp['% Después']).abs().round(2)
    
    print(f"\n> Variable: {col}")
    print(df_cat_comp.to_string(index=False))

ANÁLISIS DE SENSIBILIDAD: VARIABLES NUMÉRICAS


,Media (Antes),Media (Después),Dif. Absoluta Medias,Desv.Est (Antes),Desv.Est (Después),Mediana (Antes),Mediana (Después)
edad_materna_gest,35.58,35.58,0.00,4.71,4.71,35.71,35.71
tas_1tri,114.24,114.13,0.11,12.43,11.74,114.00,113.52
tad_1tri,73.12,73.08,0.04,9.09,8.59,73.00,73.00
eg_eco_1tri,13.04,13.04,0.00,4.72,4.51,12.80,12.80



ANÁLISIS DE SENSIBILIDAD: VARIABLES CATEGÓRICAS

> Variable: uterinas_p95_1tri
 Categoría  % Antes  % Después  Dif %
       0.0     81.9      84.01   2.11
       1.0     18.1      15.99   2.11

> Variable: riesgo_pe_1tri
 Categoría  % Antes  % Después  Dif %
       0.0    62.26      66.67   4.41
       1.0    37.74      33.33   4.41

> Variable: uterinas_p95_eco_2tri
 Categoría  % Antes  % Después  Dif %
       0.0    89.68      90.51   0.83
       1.0    10.32       9.49   0.83

> Variable: deter_sflt1_plgf_gest
 Categoría  % Antes  % Después  Dif %
       0.0    25.34      24.93   0.41
       1.0    74.66      75.07   0.41

> Variable: tomo_durante_gest
 Categoría  % Antes  % Después  Dif %
       0.0    34.24      34.15   0.09
       1.0     2.17       2.17   0.00
       2.0     1.36       1.36   0.00
       3.0    61.41      61.52   0.11
       4.0     0.54       0.81   0.27
       5.0     0.27       0.00   0.27


# Escalado

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [18]:
# El escalado es estrictamente necesario para SVM, Logistic Regression y KNN.
# Se escalan las variables numéricas y las categóricas ordinales. A las categóricas nominales se les aplica one-hot encoding. 

# Agrupamos todas las variables que se van a normalizar/escalar juntas
variables_a_escalar = num_cols + cat_cols_ord

# Diseñamos el transformador por columnas
preprocesador = ColumnTransformer(
    transformers=[
        # A las numéricas y ordinales les aplicamos StandardScaler
        ('escalador', StandardScaler(), variables_a_escalar),
        # A las nominales les aplicamos OneHotEncoder (y tiramos la primera columna para evitar colinealidad)
        ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols_nom)
    ],
    remainder='drop' # Si se colara alguna variable no registrada, se descarta por seguridad
)

X_train_scaled = preprocesador.fit_transform(train_final)
X_test_scaled = preprocesador.transform(test_final)

In [19]:
print(f"Dimensiones finales de entrenamiento: {X_train_scaled.shape}")
print(f"Dimensiones finales de test: {X_test_scaled.shape}")

Dimensiones finales de entrenamiento: (295, 12)
Dimensiones finales de test: (74, 12)


# Algoritmos de clasificación (hiperparámetros por defecto)
- Se van a aplicar los siguientes SVM, Random Forest, Decision Tree, Logistic Regressor, KNN y XGBoost con los hiperparámetros que vienen por defecto.
- Se va a evaluar el rendimiento de los algoritmos mediante la métrica f1-score.


In [20]:
# Configuración de modelos con parámetros por defecto
# Se calcula la proporción de desbalanceo para XGBoost (negativos / positivos) -> 312 / 57 ≈ 5.47
scale_pos = 312 / 57

modelos = {
    "SVM": SVC(class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42),
    "Logistic Regression": LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "KNN": KNeighborsClassifier(),  # Nota: KNN se basa en distancias matemáticas y no admite pesos de clase
    "XGBoost": XGBClassifier(scale_pos_weight=scale_pos, random_state=42, eval_metric='logloss')
}

In [21]:
# Entrenamiento, predicción y evaluación (F1-Score)
resultados = []

for nombre, modelo in modelos.items():
    # Entrenamiento del algoritmo
    modelo.fit(X_train_scaled, y_train)
    
    # Predicción sobre el conjunto de test independiente
    y_pred = modelo.predict(X_test_scaled)
    
    # Cálculo del F1-Score para la clase positiva (riesgo compuesto presente)
    score_f1 = f1_score(y_test, y_pred, average='binary')
    
    resultados.append({
        "Algoritmo": nombre,
        "F1-Score (Test)": round(score_f1, 4)
    })

### Métrica de rendimiento

In [22]:
# Presentación de resultados
df_resultados = pd.DataFrame(resultados).sort_values(by="F1-Score (Test)", ascending=False)
print("RENDIMIENTO CON HIPERPARÁMETROS POR DEFECTO (Caso 3: N=369)")
print(df_resultados.to_string(index=False))

RENDIMIENTO CON HIPERPARÁMETROS POR DEFECTO (Caso 3: N=369)
          Algoritmo  F1-Score (Test)
            XGBoost           0.4000
Logistic Regression           0.3784
                SVM           0.3333
                KNN           0.3077
      Random Forest           0.2857
      Decision Tree           0.2400


# Algoritmos de clasificación (hiperparámetros optimizados)
- Se van a aplicar los siguientes SVM, Random Forest, Decision Tree, Logistic Regressor, KNN y XGBoost con los hiperparámetros optimizados mediante GridSearch.
- Se va a evaluar el rendimiento de los algoritmos mediante la métrica f1-score.

In [23]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [24]:
ajustes_grid = {
    "SVM": {
        'model': SVC(class_weight='balanced', random_state=42, max_iter=1000000),
        'params': {
            'C': [0.1, 1, 10, 100],
            'gamma': ['scale', 'auto', 0.01, 0.1],
            'kernel': ['linear', 'rbf', 'sigmoid']
        }
    },
    "Random Forest": {
        'model': RandomForestClassifier(class_weight='balanced', random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [None, 5, 10, 15],
            'min_samples_split': [2, 5, 10],
            'criterion': ['gini', 'entropy']
        }
    },
    "Logistic Regression": {
        # Subimos max_iter a 5000 para eliminar las alertas de falta de convergencia en tu consola
        'model': LogisticRegression(class_weight='balanced', random_state=42, max_iter=5000),
        'params': {
            'C': [0.01, 0.1, 1, 10, 100],
            'penalty': ['l2'],
            'solver': ['lbfgs', 'saga']
        }
    },
    "Decision Tree": {
        'model': DecisionTreeClassifier(class_weight='balanced', random_state=42),
        'params': {
            'max_depth': [None, 3, 5, 10, 15],
            'min_samples_split': [2, 5, 10, 20],
            'criterion': ['gini', 'entropy']
        }
    },
    "KNN": {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7, 11, 15],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        }
    },
    "XGBoost": {
        'model': XGBClassifier(scale_pos_weight=(312/57), random_state=42, eval_metric='logloss'),
        'params': {
            'n_estimators': [50, 100, 150],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.01, 0.1, 0.2],
            'subsample': [0.7, 0.9, 1.0]
        }
    }
}

In [25]:
# Configuración del bucle de validación cruzada
# Usamos StratifiedKFold de 5 repartos para asegurar la misma proporción clínica en cada sub-bloque
cv_estratificado = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados_optimizados = []

print("Iniciando GridSearch para los 6 algoritmos\n")

for nombre, configuracion in ajustes_grid.items():
    print(f"Optimizando {nombre}")
    
    # Creamos el objeto GridSearch enfocado exclusivamente en maximizar el F1-score
    clf = GridSearchCV(
        estimator=configuracion['model'],
        param_grid=configuracion['params'],
        cv=cv_estratificado,
        scoring='f1',
        n_jobs=-1  
    )
    
    # Entrenamos usando tus matrices ya imputadas y escaladas en pasos anteriores
    clf.fit(X_train_scaled, y_train)
    
    # Extraemos el mejor estimador obtenido por GridSearch
    mejor_modelo = clf.best_estimator_
    
    # Evaluamos su F1-score real en el conjunto de prueba independiente (X_test_scaled)
    y_pred_test = mejor_modelo.predict(X_test_scaled)
    f1_test = f1_score(y_test, y_pred_test, average='binary')
    
    # Guardamos las métricas y los hiperparámetros ganadores
    resultados_optimizados.append({
        "Algoritmo": nombre,
        "F1-Score optimizado (Test)": round(f1_test, 4),
        "Mejores hiperparámetros": clf.best_params_
    })

Iniciando GridSearch para los 6 algoritmos

Optimizando SVM
Optimizando Random Forest
Optimizando Logistic Regression
Optimizando Decision Tree
Optimizando KNN
Optimizando XGBoost


### Métrica de rendimiento

In [26]:
# Presentación de resultados en dataFrame
df_resultados_grid = pd.DataFrame(resultados_optimizados).sort_values(by="F1-Score optimizado (Test)", ascending=False)

print("RENDIMIENTO TRAS OPTIMIZACIÓN GRIDSEARCH")
print(df_resultados_grid[["Algoritmo", "F1-Score optimizado (Test)"]].to_string(index=False))

RENDIMIENTO TRAS OPTIMIZACIÓN GRIDSEARCH
          Algoritmo  F1-Score optimizado (Test)
                SVM                      0.4186
      Random Forest                      0.4167
Logistic Regression                      0.4000
            XGBoost                      0.3636
      Decision Tree                      0.2941
                KNN                      0.1250


### Desglose de hiperparámetros óptimos

In [27]:
# Mostrar detalladamente los parámetros ganadores
print("\nDETALLE DE PARÁMETROS GANADORES")
for fila in resultados_optimizados:
    print(f"\n> {fila['Algoritmo']}:")
    print(f"  Hiperparámetros: {fila['Mejores hiperparámetros']}")


DETALLE DE PARÁMETROS GANADORES

> SVM:
  Hiperparámetros: {'C': 0.1, 'gamma': 'scale', 'kernel': 'sigmoid'}

> Random Forest:
  Hiperparámetros: {'criterion': 'entropy', 'max_depth': 5, 'min_samples_split': 10, 'n_estimators': 50}

> Logistic Regression:
  Hiperparámetros: {'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs'}

> Decision Tree:
  Hiperparámetros: {'criterion': 'gini', 'max_depth': None, 'min_samples_split': 10}

> KNN:
  Hiperparámetros: {'metric': 'euclidean', 'n_neighbors': 3, 'weights': 'distance'}

> XGBoost:
  Hiperparámetros: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.7}
